# <span style="color:#2E86C1">Memòria · BiciTierraMarket</span>

## <span style="color:#16A085">1. Plantejament del problema</span>

**Pregunta principal:** què fa que venguem més unitats: el màrqueting, els descomptes o altres factors?

- **Dades:** `sales_history.csv` (1.188 registres, 17 columnes).
- **Variable objectiu:** `sales_units` (unitats venudes).
- **Predictors principals:** `marketing_spend` i `avg_discount`.

### <span style="color:#D68910">1.1 Predictors inicials</span>

La matriu de correlació mostra quines variables tenen més relació amb les vendes.

![Matriu de correlació](./imatges/Matriu_de_correlació.png)

### <span style="color:#D68910">1.2 Qualitat de les dades</span>

- **Nuls:** es tracten segons el context. Un descompte buit = `0` (no n'hi havia); una edat buida = mitjana (una edat de 0 no té sentit).
- **Valors extrems i duplicats:** no n'hi ha en aquests fitxers.

Més detalls a [`customers.ipynb`](../customers.ipynb) i [`clean_saleshistory.ipynb`](../clean_saleshistory.ipynb).

### <span style="color:#D68910">1.3 Referència i primers models</span>

**Referència (baseline):** predir sempre la mitjana de `sales_units`. Tot model l'ha de superar.

Proves exploratòries (cada membre un model, encara no comparables del tot):

| Model | Resultat orientatiu |
|---|---|
| Regressió lineal | R² ≈ 0,90 · MAE ≈ 11 u. |
| **Regressió polinomial (grau 2)** | **R² ≈ 0,92 · MAE ≈ 10 u.** |
| Regressió logística (baix/mitjà/alt) | Accuracy ≈ 0,80 |
| Gradient Boosting | R² ≈ 0,79 · MAE ≈ 16 u. |

**Conclusions:**
- El grau 2 millora poc la lineal -> la relació és sobretot **lineal**. El grau 3 **sobreajusta** (R2 test cau de 0,92 a 0,83).
- Descartem la **classificació**: tallar les vendes en trams fa confusos els casos de frontera (només 67% d'encert en vendes mitjanes).
- Treballem amb **regressió**: la lineal com a base (fàcil d'interpretar) i la polinomial i el Gradient Boosting com a alternatives.
- **Mètriques:** MAE (error en unitats) i R2 (per comparar models).

### <span style="color:#D68910">1.4 Hipòtesis</span>

Volem l'efecte del màrqueting i del descompte **a igualtat de canal, línia de producte, regió i mes**:

| | Hipòtesi | Com es comprova |
|---|---|---|
| **H0** | Les vendes depenen sobretot del canal, la línia, la regió i el mes. | Correlació ≈ 0,85 amb la mitjana per grup |
| **H1** | Més màrqueting -> més vendes, però cada vegada guanyem menys. | Polinomial: es refuta si la corba és recta o plana |
| **H2** | Més descompte -> més vendes, amb efecte limitat. | Lineal + polinomial: es refuta si el coeficient és negatiu o no significatiu |
| **H3** | El màrqueting actua sobretot a través de les visites web (e-commerce). | Lineal amb i sense `website_sessions`: es refuta si el coeficient no canvia |

## <span style="color:#16A085">2. Preparació reproduïble i modelatge de referència</span>

**Objectiu:** establir un flux de treball reproduïble, evitar la fuga de dades (`data leakage`) i fixar unes dades mínimes (`baseline`) que els models de predicció hagen de superar.

### <span style="color:#D68910">2.1 Preparació reproduïble</span>

Per garantir que qualsevol execució del procés done exactament els mateixos resultats, s'apliquen tres regles:

- **Ordre seqüencial estricta:** Les cel·les del *notebook* s'han d'executar de dalt a baix sense bots (salts) per evitar errors d'estat o variables no definides.
- **Preservació dels fitxers originals:** Mai es modifiquen els CSV originals (`sales_history.csv`, `customers.csv`). Totes les neteges i transformacions es guarden en fitxers derivats (com per exemple `out.csv`) per mantindre la traçabilitat.
- **Semilla fixa (`seed`):** Es fixa una semilla (`seed`) aleatòria perquè les particions i operacions amb components *pseudoaleatoris* siguen sempre idèntiques.

### <span style="color:#D68910">2.2 Decisions de qualitat</span>

Abans d'entrenar cap model, s'imputen els valors nuls segons la lògica de negoci de cada variable:

| Fitxer | Variable | Tractament dels nuls | Motiu |
|---|---|---|---|
| `sales_history.csv` | `marketing_spend`, `marketing_budget` | **Mitjana** | L'empresa manté una inversió i pressupost recurrent en màrqueting; un `0` distorsionaria la realitat. |
| `sales_history.csv` | `website_sessions` | **Mitjana** | Sempre hi ha trànsit web mensual, especialment amb vendes al canal *`ecommerce`*. |
| `sales_history.csv` | `avg_discount`, `planned_discount` | **`0`** | Un valor buit indica que no n'hi havia cap descompte aplicat o planificat (venda a preu complet). |
| `customers.csv` | `age` | **Mitjana** | Un client comprador no pot tindre 0 anys; la mitjana manté la coherència demogràfica. |
| `customers.csv` | `annual_spend`, `days_since_last_order` | **`0`** | En la revisió inicial s'interpreta com a que no s'ha comprat res o una compra realitzada al mateix dia de la captura. |

> **Risc de mala imputació:** posar `0` en lloc de la mitjana a `sales_units` crearia valors extrems artificials (*outliers*), ja que cap sèrie baixa mai a zero vendes, empitjorant el MAE, l'MSE i el R2.

Més detalls al *notebook* [`prova_visualitzacions.ipynb`](../evidencies/prova_visualitzacions.ipynb).

### <span style="color:#D68910">2.3 Selecció de predictors i prevenció de fuga de dades</span>

Només es poden usar com a predictors les variables **conegudes abans que comence el mes** a predir (tal com es reflecteix al csv `sales_forecast_input.csv`):

| Variable | Estat | Motiu |
|---|---|---|
| `month`, `days_in_month` | **Admesa** | Dades de calendari les quals capturen l'estacionalitat. |
| `channel`, `product_line`, `region` | **Admesa** | Identifiquen cadascuna de les sèries temporals. |
| `stock_units`, `list_price` | **Admesa** | Estoc inicial (limita les vendes màximes) i preu de catàleg coneguts a inici de mes. |
| `marketing_budget`, `planned_discount`, `campaign` | **Admesa** | Decisions comercials planificades abans d'iniciar el mes. |
| `lag_1`, `lag_12` (i retards d'altres mètriques) | **Admesa (derivada)** | Vendes i activitat de mesos ja tancats (*`t−1`*, *`t−12`*), calculades dins de cada sèrie sense creuar-les. |
| `revenue` | **Exclosa** | **Fuga directa:** `revenue = sales_units × list_price × (1 − avg_discount)`. Permetria deduir l'objectiu de forma exacta i com a tal no existeix a inici de mes. |
| `website_sessions` | **Exclosa** | **Simultaneïtat i causalitat inversa:** s'acumulen durant el mes com a efecte de la mateixa demanda i no són planificables. |
| `marketing_spend`, `avg_discount` | **Exclosa** | Són valors reals coneguts només en tancar el mes, els quals es substitueixen pels seus equivalents planificats. |
| `closing_stock_units`, `stockout` | **Exclosa** | Depenen directament de les vendes del mes (`stock_units − sales_units`), revelant la resposta. |

### <span style="color:#D68910">2.4 Referències (Baselines) i validació</span>

Per avaluar la capacitat predictiva sense filtrar informació del futur, s'aplica una partició temporal estricta:

- **Entrenament:** 2022–2024 (2021 s'exclou per disposar de `lag_12`).
- **Validació:** 2025 (el conjunt de prova de 2026 es reserva sense tocar).
- **Restricció física:** totes les prediccions s'acoten entre `0` i `stock_units`, ja que no es pot vendre per damunt de l'estoc disponible.

Avaluem dues regles simples sobre el conjunt de validació:

| Referència | Descripció | MAE (unitats) | WAPE (%) |
|---|---|---|---|
| **Mitjana històrica de grup** | Mitjana de vendes de cada sèrie calculada només en entrenament | 18,75 | 18,06 % |
| **Mateix mes de l'any anterior (`lag_12`)** | Vendes exactes de la mateixa sèrie 12 mesos enrere | **17,40** | **16,77 %** |

**Conclusions i llistó per a la següent fase:**
- **Guanya `lag_12`:** supera la mitjana històrica perquè captura l'**estacionalitat** (pics d'estiu o Nadal).
- **Impacte en negoci:** sobre una mitjana d'aproximadament 104 unitats venudes per sèrie i mes, un MAE de 17,40 suposa un desviament mensual d'unes **17 bicicletes per sèrie** (estimadament 17 % del volum), generant risc de sobrestoc o ruptures d'estoc.
- **Objectiu de modelatge:** qualsevol model entrenat haurà de baixar d'un **MAE de 17,40 unitats** i un **WAPE del 16,77 %** en validació.

## <span style="color:#16A085">3. Segmentació de clients</span>

**Objectiu:** descriure perfils de client que ajuden a decidir campanyes, serveis o prioritats comercials.

Ens basem en `customers.csv`: una taula de 2.000 clients a 30/06/2026 (una fila = un client).

### <span style="color:#D68910">3.1 Qualitat de les dades</span>

Abans de segmentar revisem les variables de comportament (recència, freqüència, valor de la compra i devolucions) i la despesa anual.

| Aspecte | Què trobem | Decisió |
|---|---|---|
| **Nuls en `annual_spend`** (30) | La despesa anual és tiquet mitjà × comandes | La **recalculem** (la comprovació coincideix). No imputem amb 0 ni amb la mitjana |
| **Nuls en `days_since_last_order`** (20, ≈ 1 %) | Un `0` voldria dir "ha comprat hui", que és un valor real | **No posem 0**: excloem eixes 20 files del clustering (queden 1.980 clients) |
| **Nuls en `age`** (20) | No és variable de clustering | Es deixa tal qual: només serveix per a descriure els perfils |
| **Valors extrems** | `avg_order_value` té una cua llarga cap a la dreta (tiquets de més de 1.000 €) | **No els eliminem**: són els clients que més interessen a l'empresa. Apliquem `log` perquè no dominen el K-Means |
| **Recència, freqüència i devolucions** | Distribucions revisades amb resum i histogrames | Es queden com estan, sense transformar |

Més detalls a [`cp3.ipynb`](../entregues/cp3.ipynb).

### <span style="color:#D68910">3.2 Variables del clustering</span>

Usem **quatre variables**, cadascuna amb un paper diferent:

| Variable | Decisió | Motiu |
|---|---|---|
| `days_since_last_order` | **Usada** | Recència |
| `orders_last_12m` | **Usada** | Freqüència |
| `avg_order_value` (en log) | **Usada** | Valor de cada compra |
| `returns_ratio` | **Usada** | Devolucions |
| `annual_spend` | **Descartada** | És tiquet mitjà × comandes: usar les tres donaria massa pes al mateix comportament |
| `returned_orders_last_12m` | Descartada | Redundant amb `returns_ratio` i `orders_last_12m` |
| `customer_id`, `snapshot_date` | **Excloses** | Identificador i data de la fotografia: no descriuen comportament |
| `age`, `city`, `channel_preference`, `bike_interest`, `service_interest`, `tenure_months` | Només per a descriure | Les mirem després per a interpretar els perfils |

El gast anual no ens aporta informació nova: ja va dins del tiquet mitjà i de les comandes. Deixar-la fora evita contar dos vegades el mateix comportament.

### <span style="color:#D68910">3.3 K-Means</span>

**Preparació:** escalem les variables (mitjana `0` i desviació `1`) perquè cap pese més només per tindre números més grans (dies contra ràtios, per exemple). El tiquet mitjà ja va en `log`.

**Referència:** una segmentació per regles senzilles (devolucions altes, inactius, valuosos, resta). Detecta els extrems, però deixa un grup `Resta` molt gran i no separa els clients de tiquet molt alt dels de freqüència alta.

**Nombre de grups:** provem `k` de 2 a 8 i mirem la inèrcia (colze) i el *`silhouette`*.

- El colze no és clar.
- El *`silhouette`* puja fins a 5–6 i després baixa.
- Amb **`k` = 4** dos perfils importants queden barrejats (tiquet molt alt i alta freqüència).
- Amb **`k` = 6** no apareix cap perfil nou útil.

**Elecció: `k` = 5.** Cada perfil es pot explicar i accionar. El *`silhouette`* és modest (≈ 0,34): els grups no estan molt separats, i cal tindre-ho present quan es diu que un client `és` d'un segment (≈ el 3 % dels clients tenen *silhouette* negatiu).

**Conclusió:** K-Means aporta més detall que les regles, tot i que és menys fàcil d'explicar a un tercer.

### <span style="color:#D68910">3.4 Perfils</span>

Dels cinc grups, ens quedem amb **dos perfils especialment útils** per a decidir. Els valors són orientatius (medianes de cada grup).

| Perfil | Mida | Comportament | Què el diferencia |
|---|---|---|---|
| **Fidels freqüents** | ≈ 27% | Compren molt (≈ 9 comandes/any), han comprat fa poc (≈ fa 24 dies) i amb tiquet mitjà (≈ 170 €) | Més freqüència i més recència. Mostren més interés per *mountain* i *electric* |
| **Tiquet alt** | ≈ 20% | Pocs encàrrecs (≈ 5) però molt cars (≈ 620 €), amb la despesa anual més alta (≈ 2.750 €) | La major part s'interessa per *electric* (≈ 50%), i són una mica més majors i amb menys antiguitat |

**Per què són útils:** són dos camins diferents per a crear valor. Uns ho fan amb **volum de compres** i els altres amb **import per compra**. Si no els separem, tractaríem igual dos clients que necessiten coses molt diferents.

**Límits:**
- El canal preferit i l'interés per serveis són casi iguals en tots els grups, així que no ajuden a diferenciar.
- La fotografia és d'un sol dia: no sabem si un client canvia de perfil amb el temps.
- No hi ha cap etiqueta correcta: amb una altra `k` o un altre escalat els grups canviarien una mica.

### <span style="color:#D68910">3.5 Accions per perfil</span>

| Perfil | Acció proposada | Objectiu |
|---|---|---|
| **Fidels freqüents** | Programa de fidelitat amb accés anticipat a novetats i avantatges per volum en accessoris i manteniment, centrat en *mountain* i *electric* | Mantindre la freqüència i evitar que passen a inactius |
| **Tiquet alt** | Campanya personalitzada en `black_friday` o `christmas` amb bicicleta elèctrica, opcions de finançament i servei postvenda prioritari | Augmentar la taxa de resposta i protegir el valor de cada compra |

**Com ho mesurarem:** comparant un grup que rep l'acció amb un grup de control (taxa de resposta, comandes i tiquet mitjà).

**Cautela:** són propostes, no resultats. Les taxes de resposta de l'apartat de taula de segments i accions comercials de `cp3.ipynb` són simulades, així que no demostren que l'acció funcione. Cal provar-la abans d'escalar-la. A més, els clients fronterers (*silhouette* baix) no s'han de tractar com a membres segurs d'un perfil.